# Gisement de flexibilité du chauffage — simulation contrefactuelle

La consigne de thermostat est une **entrée** du réseau de `06_timeseries/timeseries_net`.
On peut donc la modifier et relire la courbe de charge : c'est une expérience contrefactuelle.

```
consignes réelles    -> f(t)  -> modèle -> courbe de référence
consignes modifiées  -> f'(t) -> modèle -> courbe scénario

        ΔP(t) = référence − scénario     ΔP > 0 : effacement    ΔP < 0 : report
```

**Ce qui rend la démarche crédible** : une partie des bâtiments du parc ont déjà un réduit de
nuit dans leurs consignes. Le réseau a donc appris sur des variations de consigne
intra-journalières réelles et leurs conséquences — ce ne sont pas des extrapolations pures.

**Limite assumée** : le modèle ne prédit pas la température intérieure, donc le confort n'est
pas vérifié. C'est pourquoi les scénarios bornent le décalage à des amplitudes du même ordre
que les réduits que ResStock applique déjà lui-même.

---

## Ce que ce notebook restitue

Le calcul lui-même vit dans `experiments/flex_chauffage.py` — 101 bâtiments × 13 scénarios,
soit une quarantaine de minutes de calcul. Ce notebook **relit le résultat** et produit les
lectures. Pour recalculer :

```bash
cd experiments
python flex_chauffage.py --sortie flex_chauffage.json
python figures_flex.py
```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RES  = Path().resolve().parent.parent / 'experiments' / 'results'
d    = json.loads((RES / 'flex_chauffage.json').read_text(encoding='utf-8'))
tab  = lambda lignes: pd.DataFrame(lignes)

print(f"{d['meta']['n_bat']} bâtiments de validation"
      f" | fenêtres en heure {'locale' if d['meta']['local'] else 'du fichier'}")
print(f"scénarios : {', '.join(d['scenarios'])}")

## 1. Deux corrections sans lesquelles les chiffres seraient faux

### 1a. Les colonnes dérivées doivent bouger avec la consigne

`f(t)` contient la consigne **et trois colonnes qui en dérivent** : les deux écarts rectifiés
et le besoin électrique de chauffage. Décaler la consigne sans recalculer les trois
présenterait au réseau une entrée physiquement incohérente.

Le piège est qu'un **test d'identité ne le verrait pas** : à décalage nul, les trois colonnes
sont inchangées de toute façon. C'est un bug qui ne se manifeste que sur les scénarios réels.

### 1b. Les fenêtres horaires sont en heure locale, pas en heure du fichier

Les séries ResStock sont horodatées dans un **fuseau unique pour tout le pays**, cohérent avec
l'heure de l'Est — et non en heure locale. Vérifié sur le bâtiment 70818 (Californie) contre
le fichier météo de son comté : moyenne annuelle identique au millième, mais le pic de
rayonnement du 15 janvier tombe à 12 h dans le fichier comté et à 14 h 30 dans la série,
soit **3 h d'écart** une fois corrigées les conventions d'étiquetage — l'écart PST → EST.

Sans correction, une fenêtre « 18–21 h » vise **15–18 h locales** en Californie. Sur le
bâtiment 70818, la pointe hivernale réelle est à 20 h locales, soit 23 h dans l'horloge du
fichier : le scénario la manquait entièrement.

Le banc applique donc `heure_fichier = heure_locale − round((longitude + 75) / 15)`.

In [ ]:
b = tab(d['scenarios']['soir']['batiments'])
print('décalages appliqués (h) :', sorted(b['decalage_h'].unique()))
print(b.groupby('decalage_h')['bldg_id'].count().rename('bâtiments').to_string())

## 2. Les scénarios

| scénario | fenêtre (heure locale) | décalage |
|---|---|---|
| Soir | 18–21 h | −2 °C |
| Matin | 7–10 h | −2 °C |
| Nuit | 0–6 h | −3 °C |
| Préchauffage | +2 °C de 16 à 18 h, puis −2 °C de 18 à 21 h | — |

Quatre indicateurs par bâtiment, tous sur l'hiver (décembre à février) :

- **pointe effacée** — le maximum de ΔP pendant la fenêtre, en kW. C'est ce qui intéresse un
  gestionnaire de réseau.
- **énergie effacée** — la somme de ΔP sur les heures d'événement.
- **rebond** — la consommation supplémentaire dans les 6 h qui suivent chaque bloc.
- **bilan net** — ΔP sommé sur **tout** l'hiver. Indispensable dès qu'un scénario agit hors de
  la fenêtre : le préchauffage paie sa facture *avant*, et le couple (effacé, rebond) le
  ferait passer pour gratuit.

In [ ]:
lignes = []
for nom, s in d['scenarios'].items():
    x = tab(s['batiments'])
    lignes.append({
        'scénario'          : s['libelle'],
        'pointe médiane kW' : x['pointe_kW_total'].median(),
        'effacé médian kWh' : x['efface_kWh_chauffage'].median(),
        'rebond médian kWh' : x['rebond_kWh_chauffage'].median(),
        'taux de report %'  : 100 * x['rebond_kWh_chauffage'].sum()
                              / max(x['efface_kWh_chauffage'].sum(), 1e-9),
        'bilan net kWh'     : x['net_kWh_chauffage'].median(),
    })
recap = pd.DataFrame(lignes).set_index('scénario').round(1)
print(recap.to_string())

### Lecture

**Le soir est le meilleur rapport effet/coût.** Pointe médiane de 2,4 kW pour un taux de
report **négatif** : l'énergie n'est pas rattrapée après, elle est simplement économisée,
parce que la nuit qui suit ne demande pas de remontée en température.

**La nuit efface le plus d'énergie** (337 kWh sur l'hiver) mais avec 23 % de report et surtout
au moment où le réseau n'en a pas besoin — l'effacement doit viser la pointe, pas le creux.

**Le préchauffage est un piège apparent.** Il affiche 79 kWh effacés et un report négatif, ce
qui semble excellent — jusqu'à regarder le bilan net. Le surcoût du préchauffage se paie
avant la fenêtre, hors de tous les compteurs classiques. C'est exactement pour cela que le
bilan net figure dans le tableau.

## 3. Le profil d'effacement

La signature d'un effacement : un creux pendant la fenêtre, une bosse de report après. C'est
la forme qui décide si l'effacement est utile au réseau ou s'il ne fait que déplacer le
problème.

In [ ]:
fenetres = {'soir': (18, 21), 'matin': (7, 10), 'nuit': (0, 6)}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)

for ax, nom in zip(axes, fenetres):
    prof = np.array([x['profil_chauffage'] for x in d['scenarios'][nom]['batiments']]).mean(0)
    h = np.arange(24)
    ax.bar(h[prof >= 0], prof[prof >= 0], color='#2a78d6', width=.82, label='effacement')
    ax.bar(h[prof < 0],  prof[prof < 0],  color='#eb6834', width=.82, label='report')
    h0, h1 = fenetres[nom]
    ax.axvspan(h0 - .5, h1 - .5, color='#0E6B76', alpha=.09)
    ax.axhline(0, color='black', lw=.9)
    ax.set(title=d['scenarios'][nom]['libelle'], xlabel='heure locale')
    ax.set_xticks(range(0, 24, 3)); ax.grid(axis='y', alpha=.3)

axes[0].set_ylabel('effacement moyen (kWh/h)')
axes[0].legend()
plt.suptitle('Profil moyen d\'effacement — 101 bâtiments de validation, hiver', fontsize=13)
plt.tight_layout(); plt.show()

## 4. Jusqu'où peut-on pousser ?

Deux questions distinctes : augmenter **l'amplitude** du décalage, ou allonger la **durée** de
la fenêtre. Les deux ne se comportent pas pareil.

In [ ]:
abs_f = lambda k: abs(float(k))   # trier 1,2,3,4 et non -4,-3,-2,-1
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, source, xs, xlabel in [
        (axes[0], d['amplitudes'], [abs(float(k)) for k in sorted(d['amplitudes'], key=abs_f)],
         'amplitude du décalage (°C)'),
        (axes[1], d['durees'], [float(k) for k in sorted(d['durees'], key=float)],
         'durée de la fenêtre (h)')]:
    cles = sorted(source, key=abs_f)
    med = [tab(source[c])['efface_kWh_chauffage'].median() for c in cles]
    q1  = [tab(source[c])['efface_kWh_chauffage'].quantile(.25) for c in cles]
    q3  = [tab(source[c])['efface_kWh_chauffage'].quantile(.75) for c in cles]
    ax.fill_between(xs, q1, q3, color='#2a78d6', alpha=.16)
    ax.plot(xs, med, 'o-', color='#2a78d6', lw=2, ms=7)
    for x, m in zip(xs, med):
        ax.annotate(f'{m:.0f}', (x, m), textcoords='offset points', xytext=(0, 10),
                    ha='center', fontsize=9)
    ax.set(xlabel=xlabel, ylabel='énergie effacée sur l\'hiver (kWh)', ylim=(0, None))
    ax.set_xticks(xs); ax.grid(alpha=.3)

axes[0].set_title('Rendements décroissants sur l\'amplitude')
axes[1].set_title('Réponse quasi linéaire à la durée')
plt.tight_layout(); plt.show()

amp = sorted(d['amplitudes'], key=abs_f)
m = [tab(d['amplitudes'][c])['efface_kWh_chauffage'].median() for c in amp]
print('gain marginal par degré supplémentaire :')
for i in range(1, len(m)):
    print(f'  {abs(float(amp[i-1])):.0f} → {abs(float(amp[i])):.0f} °C : +{m[i]-m[i-1]:.0f} kWh')

### Lecture

**L'amplitude sature.** Passer de 1 à 2 °C rapporte +35 kWh, mais de 3 à 4 °C seulement
+14 kWh. La raison est physique : au-delà d'un certain décalage, le chauffage est déjà à
l'arrêt pendant toute la fenêtre — baisser davantage la consigne ne peut plus rien couper.
Le gain plafonne à la consommation qu'il y avait à effacer.

**La durée, elle, reste presque linéaire** jusqu'à 6 h. Chaque heure supplémentaire ajoute à
peu près le même effacement, parce qu'on couvre de nouvelles heures de chauffe.

**Conséquence pratique** : pour augmenter un gisement, allonger la fenêtre est plus efficace
que creuser l'amplitude — et c'est aussi plus confortable pour l'occupant, puisque la
température dérive moins vite.

## 5. Quels logements recruter

Un gisement médian ne dit rien de la stratégie de recrutement. Il faut savoir **où** il se
concentre.

In [ ]:
b = tab(d['scenarios']['soir']['batiments'])

print('Par système de chauffage')
print(b.groupby(b['pac'].map({True: 'pompe à chaleur', False: 'résistance'}))
        [['pointe_kW_total', 'efface_kWh_chauffage']].median().round(2).to_string())

print('\nCorrélations de Spearman avec la pointe effacée')
for c, lib in [('tau', 'constante de temps τ'), ('ua', 'déperditions UA')]:
    print(f'  {lib:24} {b[c].corr(b["pointe_kW_total"], method="spearman"):+.2f}')

v = np.sort(b['pointe_kW_total'].values)[::-1]
part = np.cumsum(v) / v.sum() * 100
for seuil in (10, 20, 30, 50):
    print(f'\n{seuil:2d} % des logements les plus effaçables '
          f'→ {part[int(len(v)*seuil/100)-1]:.0f} % du gisement')

### Lecture

Le gisement est **fortement concentré** : 30 % des logements portent plus de la moitié du
total. Un programme d'effacement n'a donc aucun intérêt à recruter au hasard.

En revanche, les critères de recrutement évidents fonctionnent mal. Les résistances
électriques effacent un peu plus que les pompes à chaleur — logique, puisqu'elles consomment
trois fois plus pour le même besoin — mais les corrélations avec la constante de temps et
avec les déperditions sont **faibles** (+0,12 et +0,17). Autrement dit, on ne sait pas encore
prédire quels logements sont les plus effaçables à partir de leurs caractéristiques
d'enveloppe seules. C'est la principale limite ouverte de cette étude.

## 6. Contrôles de validité

Deux tests que le banc exécute avant toute mesure, et dont le résultat est rappelé ici.

**Test d'identité** — à décalage nul, la prédiction doit être inchangée. L'écart mesuré est de
3,8 × 10⁻⁶ kWh : ce n'est pas exactement zéro parce que les colonnes dérivées sont recalculées
en float32 alors que le cache les stocke après un passage en float64. C'est un arrondi
machine, sans conséquence sur des grandeurs de l'ordre du kWh.

**Test négatif** — décaler la consigne de chauffage en plein été ne doit rien changer à
l'hiver. Mesuré : −0,000 kWh. Le modèle n'invente pas d'effet là où la physique l'interdit.

---

## 7. Ce qui reste ouvert

1. **Le confort n'est pas vérifié.** Le modèle ne prédit pas la température intérieure. Un
   gisement sans contrainte de confort n'est pas publiable tel quel — il faudrait au minimum
   un indicateur substitut construit sur `tau` et `UA`.
2. **Le foisonnement n'est pas traité.** Ces profils supposent que tous les logements effacent
   à la même heure locale. Un rebond synchronisé peut créer une pointe pire que celle qu'on
   efface ; décaler les groupes de 30 minutes est la parade classique, non testée ici.
3. **La climatisation et l'eau chaude restent à faire.** Pour la climatisation le protocole est
   symétrique. Pour l'eau chaude il ne l'est pas : `f(t)` contient les schedules de *puisage*,
   pas la consigne du ballon. On ne peut donc pas simuler un chauffe-eau piloté — seulement
   déplacer la demande, ce qui suppose que l'occupant change ses habitudes.